In [ ]:
# Utility
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tqdm
import time
import joblib
import shap

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

Load the Model

In [ ]:
# DEFINE PIPELINE

class SafeColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Only drop columns that actually exist
        cols_to_drop = [col for col in self.columns if col in X.columns]
        return X.drop(columns=cols_to_drop)

class DTypeCaster(BaseEstimator, TransformerMixin):
    def __init__(self, cast_map):
        # cast_map = {'column_name': 'dtype'}
        self.cast_map = cast_map

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col, dtype in self.cast_map.items():
            if col in X.columns:
                X[col] = X[col].astype(dtype)
        return X

In [ ]:
pipeline = joblib.load('/content/churn_pipeline.pkl')
print(pipeline)

Pipeline(steps=[('dtype_cast', DTypeCaster(cast_map={'HasFeedback': 'str'})),
                ('imputer',
                 ColumnTransformer(force_int_remainder_cols='deprecated',
                                   remainder='passthrough',
                                   transformers=[('fill_sentiment_num',
                                                  SimpleImputer(fill_value=3,
                                                                strategy='constant'),
                                                  ['sentiment_num_roberta']),
                                                 ('fill_sentiment_score',
                                                  SimpleImputer(fill_value=0.0,
                                                                strategy='constant')...
                 SafeColumnDropper(columns=['Online Security_No internet '
                                            'service',
                                            'Device Protection_No internet

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking c

In [ ]:
preprocessor = pipeline[:-1]
print(preprocessor)

Pipeline(steps=[('dtype_cast', DTypeCaster(cast_map={'HasFeedback': 'str'})),
                ('imputer',
                 ColumnTransformer(force_int_remainder_cols='deprecated',
                                   remainder='passthrough',
                                   transformers=[('fill_sentiment_num',
                                                  SimpleImputer(fill_value=3,
                                                                strategy='constant'),
                                                  ['sentiment_num_roberta']),
                                                 ('fill_sentiment_score',
                                                  SimpleImputer(fill_value=0.0,
                                                                strategy='constant')...
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7ff2a8ae1430>)],
                                   verbose_feature_names_out=False)

In [ ]:
model = pipeline[-1]
print(model)

RandomForestClassifier(class_weight='balanced', min_samples_split=5,
                       n_estimators=300, random_state=42)


Loading Data

In [ ]:
data = pd.read_csv('/content/data_with_churn_score.csv')
data = data.drop(columns=['Churn Score'], axis=1)
data.head()

,CustomerID,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,...,Unlimited Data,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Satisfaction Score,CustomerFeedback,HasFeedback,sentiment_label_roberta,sentiment_score_roberta,sentiment_num_roberta
0,3668-QPYBK,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,...,Yes,0.0,0,20.94,1,NaN,False,NaN,0.000000,3.0
1,9237-HQITU,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,...,Yes,0.0,0,18.24,2,NaN,False,NaN,0.000000,3.0
2,9305-CDSKC,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,...,Yes,0.0,0,97.20,3,NaN,False,NaN,0.000000,3.0
3,7892-POOKP,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,...,Yes,0.0,0,136.92,3,i recently decided to cancel my service after ...,True,negative,0.859597,0.0
4,0280-XJGEX,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,...,Yes,0.0,0,2172.17,1,NaN,False,NaN,0.000000,3.0


In [ ]:
df_val = pd.read_csv('/content/Telco_customer_churn.csv')

data['Churn Value'] = df_val['Churn Value']

In [ ]:
target = 'Churn Value'
y = data[target]
X = data.drop([target], axis=1)

X.head()

,CustomerID,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,...,Unlimited Data,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Satisfaction Score,CustomerFeedback,HasFeedback,sentiment_label_roberta,sentiment_score_roberta,sentiment_num_roberta
0,3668-QPYBK,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,...,Yes,0.0,0,20.94,1,NaN,False,NaN,0.000000,3.0
1,9237-HQITU,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,...,Yes,0.0,0,18.24,2,NaN,False,NaN,0.000000,3.0
2,9305-CDSKC,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,...,Yes,0.0,0,97.20,3,NaN,False,NaN,0.000000,3.0
3,7892-POOKP,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,...,Yes,0.0,0,136.92,3,i recently decided to cancel my service after ...,True,negative,0.859597,0.0
4,0280-XJGEX,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,...,Yes,0.0,0,2172.17,1,NaN,False,NaN,0.000000,3.0


In [ ]:
seed = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    shuffle=True, stratify=y,
                                                    random_state=seed)



In [ ]:
# Transform your background data
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

ValueError: The dtype of the filling value (i.e. dtype('O')) cannot be cast to the input data that is dtype('float64'). Make sure that the dtypes of the input data is of the same kind between fit and transform.

In [ ]:
import shap
import pandas as pd
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import numpy as np

# Separate features (X) from non-feature columns
# Retain 'CustomerID' temporarily because a ColumnTransformer in the pipeline expects it.
X_data_for_pipeline = data.drop(columns=['Churn Score'])

# Explicitly ensure 'sentiment_num_roberta' is numeric before preprocessing
# This step handles potential mixed types and ensures consistency for the imputer.
if 'sentiment_num_roberta' in X_data_for_pipeline.columns:
    X_data_for_pipeline['sentiment_num_roberta'] = pd.to_numeric(X_data_for_pipeline['sentiment_num_roberta'], errors='coerce')
    X_data_for_pipeline['sentiment_num_roberta'] = X_data_for_pipeline['sentiment_num_roberta'].astype(float)

# Create a sub-pipeline for all preprocessing steps except the final RandomForestClassifier
preprocessing_pipeline = Pipeline(model.steps[:-1])

# Get the actual classifier from the pipeline
classifier = model.named_steps['model']

# Transform the raw data through the preprocessing pipeline. CustomerID is present here.
X_preprocessed_with_id = preprocessing_pipeline.transform(X_data_for_pipeline)

# --- Get Feature Names After Preprocessing ---
# This involves getting names from the main preprocessor (ColumnTransformer) and then accounting for the SafeColumnDropper.

# Get feature names from the 'preprocessor' step in the model pipeline
preprocessor_ct = model.named_steps['preprocessor']
feature_names_after_preprocessor_full = preprocessor_ct.get_feature_names_out()

# Identify columns that `SafeColumnDropper` would drop
column_dropper_step = model.named_steps['column_dropper']
columns_to_drop_by_dropper = column_dropper_step.columns

# Filter the feature names to get the set that matches X_preprocessed_with_id (before removing CustomerID)
final_feature_names_with_id = [f for f in feature_names_after_preprocessor_full if f not in columns_to_drop_by_dropper]

# Now, identify and remove 'CustomerID' from the features and the preprocessed data for SHAP
if 'CustomerID' in final_feature_names_with_id:
    customer_id_idx = final_feature_names_with_id.index('CustomerID')
    print(f"Found 'CustomerID' at index {customer_id_idx} in preprocessed features. Removing it for SHAP.")

    # Remove 'CustomerID' from feature names list
    final_feature_names = [f for i, f in enumerate(final_feature_names_with_id) if i != customer_id_idx]

    # Remove the corresponding column from X_preprocessed_with_id numpy array
    X_preprocessed = np.delete(X_preprocessed_with_id, customer_id_idx, axis=1)
else:
    print("'CustomerID' not found in preprocessed features. Proceeding with all features.")
    final_feature_names = final_feature_names_with_id
    X_preprocessed = X_preprocessed_with_id

# Initialize the SHAP TreeExplainer with the trained RandomForestClassifier
explainer = shap.TreeExplainer(classifier)

# Calculate SHAP values for the now-cleaned preprocessed data
shap_values = explainer.shap_values(X_preprocessed)

print(f"SHAP values calculated. Shape: {len(shap_values)} classes, {shap_values[0].shape[0]} samples, {shap_values[0].shape[1]} features.")

# Determine the index for the positive class (churn)
# Assuming '1' is the positive class, its index can be found in classifier.classes_
try:
    churn_class_idx = list(classifier.classes_).index(1)
    shap_values_for_churn = shap_values[churn_class_idx]
    print(f"SHAP values for 'churn' class (class {classifier.classes_[churn_class_idx]}) selected.")
except ValueError:
    print("Could not find class '1' in model.classes_. Using the first set of SHAP values.")
    shap_values_for_churn = shap_values[0] # Fallback to first class if '1' not explicitly found

print(f"Number of final feature names for SHAP: {len(final_feature_names)}")
print(f"Number of features in X_preprocessed for SHAP: {X_preprocessed.shape[1]}")

# Assert that the number of feature names matches the number of features in X_preprocessed
if len(final_feature_names) != X_preprocessed.shape[1]:
    print("Warning: Mismatch between number of derived feature names and preprocessed features for SHAP. SHAP plots might have incorrect labels.")
    # Fallback to generic names if mismatch, to avoid errors
    final_feature_names = [f"feature_{i}" for i in range(X_preprocessed.shape[1])]

# Visualize the SHAP values using a summary plot for the churn class
print("\nGenerating SHAP summary plot for churn class...")
shap.summary_plot(shap_values_for_churn, X_preprocessed, feature_names=final_feature_names)
plt.show() # Display the plot

# Store the SHAP values in a variable for potential future use
shap_values_df = pd.DataFrame(shap_values_for_churn, columns=final_feature_names)
print("\nSHAP values for churn class stored in 'shap_values_df' DataFrame.")

ValueError: The dtype of the filling value (i.e. dtype('O')) cannot be cast to the input data that is dtype('float64'). Make sure that the dtypes of the input data is of the same kind between fit and transform.